# Ollama에서 Hugging Face GGUF 모델 사용하기

Hugging Face에 공개된 GGUF 모델은 파일을 별도 폴더에 내려받고 `Modelfile`로 등록하지 않아도 `hf.co/{사용자}/{저장소}:{양자화}` 형식으로 Ollama에 가져올 수 있다. 이 노트북에서는 모델을 Ollama cache에 한 번 준비한 뒤 Python API와 LangChain에서 같은 model ID를 재사용한다.


## GGUF 포맷

[GGUF](https://huggingface.co/docs/hub/en/gguf)는 모델 가중치뿐 아니라 tokenizer와 실행에 필요한 metadata를 하나의 파일에 담는 포맷이다. llama.cpp와 Ollama 같은 경량 추론 엔진이 빠르게 읽을 수 있으며, 여러 양자화 방식을 지원한다.

- **단일 파일**: 가중치와 metadata를 함께 보관해 배포하기 쉽다.
- **양자화 지원**: 4비트·5비트처럼 낮은 정밀도로 저장해 파일 크기와 추론 메모리를 줄일 수 있다.
- **실행 엔진 호환**: llama.cpp 계열 도구와 Ollama에서 사용할 수 있다.

`Q5_K_M`은 5비트 계열 양자화 방식이다. 일반적으로 더 낮은 bit 수는 메모리를 줄이지만 원본 가중치와의 차이가 커질 수 있다.

## 선수 조건과 패키지 준비

RunPod에서 `01_ollama.ipynb`를 먼저 완료해 Ollama server가 실행 중이어야 한다. Python의 `ollama` package는 새 server를 만드는 도구가 아니라 같은 Pod의 `http://localhost:11434` server에 요청하는 client이다.

In [1]:
from statistics import mode
%pip install -U ollama langchain-ollama

import ollama
from langchain_ollama import ChatOllama


  Using cached ollama-0.6.2-py3-none-any.whl.metadata (5.8 kB)
  Using cached pydantic-2.13.4-py3-none-any.whl.metadata (109 kB)
  Using cached annotated_types-0.8.0-py3-none-any.whl.metadata (15 kB)
  Using cached pydantic_core-2.46.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.6 kB)
  Using cached typing_inspection-0.4.4-py3-none-any.whl.metadata (2.6 kB)
Using cached ollama-0.6.2-py3-none-any.whl (15 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 570.0/570.0 kB 1.5 MB/s  0:00:00m eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 744.6/744.6 kB 3.9 MB/s  0:00:00
Using cached pydantic-2.13.4-py3-none-any.whl (472 kB)
Using cached pydantic_core-2.46.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (2.1 MB)
Using cached annotated_types-0.8.0-py3-none-any.whl (13 kB)
Using cached typing_inspection-0.4.4-py3-none-any.whl (14 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 19.1 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## Hugging Face GGUF 모델 바로 사용하기

`heegyu/EEVE-Korean-Instruct-10.8B-v1.0-GGUF:latest` 저장소는 현재 Ollama의 Hugging Face manifest 처리에서 호환 오류가 발생한다. 따라서 같은 `yanolja/EEVE-Korean-Instruct-10.8B-v1.0` 기반의 `Q5_K_M` GGUF인 [SourPineapple 저장소](https://huggingface.co/SourPineapple/EEVE-Korean-Instruct-10.8B-v1.0-Q5_K_M-GGUF)를 사용한다.

`ollama pull`은 GGUF를 임의의 `/workspace` 폴더에 저장하는 명령이 아니다. Ollama가 관리하는 model cache에 최초 한 번 내려받아 이후 `generate()`와 `chat()`이 같은 model ID를 사용할 수 있게 한다. [Hugging Face의 Ollama 가이드](https://huggingface.co/docs/hub/en/ollama)는 저장소 뒤에 `:Q5_K_M`처럼 tag를 붙여 원하는 양자화를 선택하는 형식을 제공한다.

In [2]:
MODEL_ID = (
    'hf.co/SourPineapple/'
    'EEVE-Korean-Instruct-10.8B-v1.0-Q5_K_M-GGUF:Q5_K_M'
)


### Ollama cache에 모델 준비하기

Python client의 `pull()`에 model ID를 전달하면 Ollama server가 Hugging Face에서 GGUF를 내려받아 자체 cache로 관리한다. 최초 실행은 파일 크기만큼 시간이 필요하지만 이후 실행에서는 저장된 layer를 재사용한다.

In [3]:
import ollama

pull_result = ollama.pull(MODEL_ID)
print(pull_result)

status='success' completed=None total=None digest=None


## `ollama.generate()`로 한 번 생성하기

`generate()`는 하나의 prompt를 전달해 assistant 역할 구분 없이 텍스트를 생성한다. `model`과 `prompt`를 전달하며, 반환값의 `response`에 생성된 본문이 들어 있다.

In [5]:
response = ollama.generate(
    model = MODEL_ID,
    prompt = "Meta 회사의 Ollama가 뭐야?"
)
print(response["response"])

Meta의 Olama란, Meta, Inc.에서 개발한 가상 비서 서비스입니다. 이 서비스는 사용자들이 다양한 작업을 수행하도록 도움을 주고자 만들어졌으며, 질문 응답, 정보 검색, 알림 설정, 개인화된 추천 등 다양한 기능을 제공합니다. Olama는 머신러닝 알고리즘과 자연어 처리 기술을 사용하여 사용자의 요구와 선호도를 이해하고 그에 따라 개인화된 서비스를 제공합니다.

Meta의 Olama는 페이스북 메신저와 같은 Meta의 다양한 플랫폼과 앱을 통해 접근할 수 있습니다. 이는 사용자가 플랫폼을 떠나지 않고도 다양한 작업을 쉽고 편리하게 수행할 수 있도록 설계되었습니다. Olama는 지속적으로 학습하고 발전하여 사용자의 요구와 기대에 부응하는 보다 정확하고 관련성 있는 서비스를 제공할 수 있도록 개선됩니다.

Meta의 Olama와 유사한 다른 가상 비서 서비스로는 아마존의 알렉사, 구글의 어시스턴트, 애플의 시리 등이 있습니다. 이러한 서비스들은 사용자의 필요와 선호에 맞는 개인화된 보조 역할을 제공함으로써 다양한 작업과 상호작용을 돕는 것을 목표로 하고 있습니다. Meta의 Olama는 페이스북 메신저와 같은 Meta의 기존 플랫폼에 통합되어 사용자들이 Meta 생태계 안에서 원활한 사용자 경험을 할 수 있도록 합니다.


## `ollama.chat()`으로 역할이 있는 대화하기

`chat()`은 `system`, `user`, `assistant` 역할이 있는 message 목록을 전달한다. 반환값의 `message.content`에서 assistant 답변을 꺼낸다.

In [7]:
chat_response = ollama.chat(
    model = MODEL_ID,
    messages = [
        {
            "role": "system",
            "content:": "너는 현업 AI 엔지니어로 일하고 있고, 조언을 해주는 역할이야. 관련 질문에 대해서 간략하게 2문단 이내로 대답해."
        },
        {
            "role": "user",
            "content": "신입 AI 엔지니어가 되려면 무엇을 준비해야 해?"
        }
    ]
)
print(chat_response["message"]["content"])

신입 AI 엔지니어가 되기 위해서는 다양한 기술, 지식, 그리고 경험을 습득해야 합니다. 여기 과정을 시작하는 데 도움이 될 기본 지침들을 소개합니다:

1. 학업 배경: 컴퓨터 과학, 컴퓨터 공학, 전기 공학, 수학 또는 관련 분야의 학사 학위 취득을 고려하세요. 이러한 학위들은 인공지능의 다양한 측면을 이해하는 데 필요한 튼튼한 기초를 제공합니다. 고급 프로그래밍 언어, 데이터 구조, 알고리즘, 선형 대수, 확률 이론, 최적화, 머신러닝에 대한 지식을 습득하세요.

2. 프로그래밍 언어 숙련: Python, C++, Java, C, R과 같은 여러 프로그래밍 언어에 능숙해집니다. 이러한 언어들은 AI 프로젝트에서 자주 사용되며, 개발, 테스트 및 배포 과정을 용이하게 해줍니다.

3. 머신러닝 및 딥러닝: 신경망, 결정 트리, 강화 학습과 같은 머신러닝 기술을 이해합니다. TensorFlow, PyTorch, Scikit-learn, Keras와 같은 인기 있는 머신러닝 및 딥러닝 라이브러리를 숙련하세요.

4. 자연어 처리(NLP): NLP를 이해하여 언어 관련 AI 시스템을 구축하세요. NLTK, spaCy, Gensim과 같은 NLP 라이브러리를 익숙하게 하고, 토큰화, 의존성 파싱, 감정 분석, 텍스트 분류 같은 기술을 공부하세요.

5. 데이터 분석: AI 프로젝트는 대개 방대한 데이터 세트와 관련됩니다. 데이터 조작, 시각화, 분석을 위한 강력한 데이터 분석 기술을 개발하세요. pandas, NumPy, Matplotlib과 같은 데이터 분석 라이브러리를 숙련하세요.

6. 도메인 지식: 특정 AI 응용 프로그램을 개발하려면 금융, 보건, 헬스케어, 교육 등 특정 산업에 대한 지식을 습득하세요. 관련 분야의 최신 추세와 발전에 대해 최신 상태를 유지하세요.

7. 연구와 학습: 인공지능과 머신러닝에 관한 새로운 논문, 기사, 연구 작업을 꾸준히 최신 상태로 유지하세요. AI 포럼, 웨비나, 컨퍼런스에 참여하여 최신 산업 동향을 파악하세요.

8.

## LangChain `ChatOllama`로 같은 모델 호출하기

`ChatOllama`는 같은 Ollama server와 model ID를 LangChain의 Chat Model 인터페이스로 감싼다. GGUF를 다시 내려받거나 다른 모델로 변환하는 과정과 관계가 없다.

In [8]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model = MODEL_ID,
    temperature = 0.2,
)

langchain_response = llm.invoke("대한민국 24절기에 대해서 간단히 설명해줘.")

print(langchain_response.content)


대한민국 24절기는 우리 조상들이 자연의 변화와 농사의 중요한 시기를 파악하기 위해 정한 전통적인 절기입니다. 이들은 24개의 주요한 날들로 구성되어 있으며, 1년 내내 계절의 변화를 나타냅니다.

24절기는 2개의 큰 그룹으로 나뉩니다:

1. 24절기의 24개 절기:

   a. 24절기의 24개 절기:
      1. 입춘(立春) - 겨울의 시작, 음력 1월 7일경
      2. 우수(雨水) - 비의 시작, 음력 2월 19일경
      3. 경칩(驚蟄) - 겨울잠에서 깨어나는 시기, 음력 3월 5일경
      4. 청명(淸明) - 봄의 시작, 음력 4월 5일경
      5. 곡우(穀雨) - 봄비, 음력 4월 20일경
      6. 입하(立夏) - 여름의 시작, 음력 5월 5일경
      7. 소만(小滿) - 여름의 시작, 음력 5월 21일경
      8. 망종(芒種) - 보리 수확, 음력 6월 5일경
      9. 하지(夏至) - 여름의 정점, 음력 6월 21일경
      10. 소서(小暑) - 여름의 시작, 음력 7월 7일경
      11. 대서(大暑) - 여름의 시작, 음력 7월 23일경
      12. 입추(立秋) - 가을의 시작, 음력 8월 7일경
      13. 처서(處暑) - 여름이 끝나는 시기, 음력 8월 23일경
      14. 백로(白露) - 이슬이 맺히는 시기, 음력 9월 8일경
      15. 추분(秋分) - 가을의 시작, 음력 9월 23일경
      16. 한로(寒露) - 추위가 시작되는 시기, 음력 10월 8일경
      17. 상강(霜降) - 서리가 내리는 시기, 음력 10월 23일경
      18. 입동(立冬) - 겨울의 시작, 음력 11월 7일경
      19. 소설(小雪) - 첫눈, 음력 11월 22일경
      20. 대설(大雪) - 큰 눈, 음력 12월 7일경
      21. 동지(冬至) - 겨울의 정점, 음력 12월 22일경
      22. 소한(小寒) - 작은 추위, 음력 1월 5

## 정리

- 기본 실습은 Hugging Face model ID를 Ollama cache에 준비하고 `ollama.generate()`로 호출한다.
- `/workspace`에 GGUF를 따로 내려받는 과정은 기본 실습에서 제외한다.
- `generate()`, `chat()`, `ChatOllama`는 같은 Ollama server와 같은 model을 서로 다른 입력 형식으로 호출한다.
- `Modelfile`은 공개 GGUF의 template나 parameter를 직접 바꿔야 할 때만 선택한다.